In [0]:
df_bronze = spark.read.table("investment_pyspark.bronze.mutual_fund_holdings_raw")
df_bronze.show()

In [0]:
df_bronze.printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType, IntegerType
windowSpec = Window.partitionBy("Instrument").orderBy(F.col("_ingestion_timestamp").desc())

In [0]:
df_silver = (df_bronze
  .withColumn("Instrument", F.upper(F.trim(F.col("Instrument"))))
  .withColumn("row_num", F.row_number().over(windowSpec))
  .filter(F.col("row_num") == 1)
  .drop("row_num")
  .withColumn("Qty",F.col("Qty").cast(DecimalType(18,2)))
  .withColumn("Avg_cost",F.col("Avg_cost").cast(DecimalType(18,2)))
  .withColumn("Current_price",F.col("LTP").cast(DecimalType(18,2)))
  .withColumn("Invested_value",F.col("Invested").cast(DecimalType(18,2)))
  .withColumn("Current_value",F.col("Cur_val").cast(DecimalType(18,2)))
  .withColumn("Unrealized_pnl",F.col("P_L").cast(DecimalType(18,2)))
  .withColumn("Net_change_pct",F.col("Net_chg").cast(DecimalType(18,2)))
  .withColumn("Day_change_pct",F.col("Day_chg").cast(DecimalType(18,2)))
  .withColumn("asset_class",F.lit("MTF"))
  .withColumn("silver_processed_timestamp",F.current_timestamp())
  .filter(F.col("Instrument").isNotNull()
          & (F.col("Qty") > 0)
          & (F.col("Invested_value") > 0)
  ).select("Instrument","asset_class","Qty","Avg_cost","Current_price","Invested_value","Current_value","Unrealized_pnl","Net_change_pct","Day_change_pct","silver_processed_timestamp"
  )
)

In [0]:
df_silver.printSchema()

In [0]:
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("investment_pyspark.silver.mutual_fund_holdings_clean")

In [0]:
%sql
select * from investment_pyspark.silver.mutual_fund_holdings_clean;